# Zahlenduell – Reinforcement Learning mit Self-Play

**Lernziele:** In diesem Notebook lernen wir, wie ein Agent durch *Reinforcement Learning* ein Strategiespiel erlernt – ganz ohne explizite Regeln zu kennen.

---

## Das Spiel: Zahlenduell

**Zwei Spieler, fünf Runden, ein Gewinner.**

Jeder Spieler hat die Karten **1, 2, 3, 4, 5** auf der Hand. Jede Runde wählen beide gleichzeitig eine Karte:

| Situation | Gewinner |
|-----------|----------|
| Spieler A hat höhere Zahl | Spieler A |
| **Sonderregel:** Unterschied = 4 (z.B. 1 vs 5) | Spieler mit der *kleineren* Zahl |
| Gleichstand | Kein Punkt |

Gespielte Karten kommen **nicht** zurück. Nach 5 Runden gewinnt, wer mehr Punkte hat.

### Probiert es aus!
Spielt das Spiel kurz zu zweit durch. Was ist eure Strategie?

---
## Teil 1: Die Spielumgebung

In [ ]:
# Imports
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
from itertools import combinations

print("✅ Bibliotheken geladen")

In [ ]:
def bewerte_runde(karte_a, karte_b):
    """
    Wertet eine Runde aus.
    Gibt zurück: +1 wenn A gewinnt, -1 wenn B gewinnt, 0 bei Gleichstand.
    """
    if karte_a == karte_b:
        return 0
    # Sonderregel: Abstand von 4 → die kleinere Zahl gewinnt
    if abs(karte_a - karte_b) == 4:
        return 1 if karte_a < karte_b else -1
    # Normalfall: höhere Zahl gewinnt
    return 1 if karte_a > karte_b else -1


# Übersichtstabelle aller möglichen Rundenausgänge
print("Rundenauswertung – Ergebnis aus Sicht von Spieler A")
print(f"{'':6}", end="")
for b in range(1, 6):
    print(f"  B={b}", end="")
print()
for a in range(1, 6):
    print(f"A={a}  ", end="")
    for b in range(1, 6):
        r = bewerte_runde(a, b)
        symbol = " +1" if r == 1 else (" -1" if r == -1 else "  0")
        print(f"  {symbol}", end="")
    print()
print("\n+1 = A gewinnt Runde, -1 = B gewinnt, 0 = Unentschieden")

In [ ]:
class ZahlenduellUmgebung:
    """
    Die Spielumgebung für Zahlenduell.
    
    Zustand (State): (eigene_restkarten_als_tupel, gegnerische_restkarten_als_tupel)
    Aktion (Action): eine der eigenen Restkarten spielen
    """

    def reset(self):
        """Startet ein neues Spiel. Gibt den Anfangszustand für beide Spieler zurück."""
        self.karten_a = list(range(1, 6))  # [1, 2, 3, 4, 5]
        self.karten_b = list(range(1, 6))
        self.punkte_a = 0
        self.punkte_b = 0
        self.runde = 0
        return self._zustand_a(), self._zustand_b()

    def _zustand_a(self):
        """Zustand aus Sicht von Spieler A."""
        return (tuple(sorted(self.karten_a)), tuple(sorted(self.karten_b)))

    def _zustand_b(self):
        """Zustand aus Sicht von Spieler B (eigene Karten zuerst!)."""
        return (tuple(sorted(self.karten_b)), tuple(sorted(self.karten_a)))

    def schritt(self, aktion_a, aktion_b):
        """
        Führt eine Runde durch.
        
        Returns:
            zustand_a, zustand_b,
            belohnung_a, belohnung_b,
            fertig (bool)
        """
        assert aktion_a in self.karten_a, f"Ungültige Karte {aktion_a} für Spieler A!"
        assert aktion_b in self.karten_b, f"Ungültige Karte {aktion_b} für Spieler B!"

        # Karten entfernen
        self.karten_a.remove(aktion_a)
        self.karten_b.remove(aktion_b)
        self.runde += 1

        # Runde auswerten
        ergebnis = bewerte_runde(aktion_a, aktion_b)

        if ergebnis == 1:
            self.punkte_a += 1
            belohnung_a, belohnung_b = 1, -1
        elif ergebnis == -1:
            self.punkte_b += 1
            belohnung_a, belohnung_b = -1, 1
        else:
            belohnung_a, belohnung_b = 0, 0

        fertig = self.runde == 5

        # Endbelohnung: Gesamtspiel
        if fertig:
            if self.punkte_a > self.punkte_b:
                belohnung_a += 3
                belohnung_b -= 3
            elif self.punkte_b > self.punkte_a:
                belohnung_a -= 3
                belohnung_b += 3

        return self._zustand_a(), self._zustand_b(), belohnung_a, belohnung_b, fertig


print("✅ Umgebung definiert")
print("\nTest – eine zufällige Partie:")
env = ZahlenduellUmgebung()
za, zb = env.reset()
for r in range(5):
    ka = random.choice(env.karten_a)
    kb = random.choice(env.karten_b)
    za, zb, bel_a, bel_b, fertig = env.schritt(ka, kb)
    print(f"  Runde {r+1}: A spielt {ka}, B spielt {kb}  → Belohnung A={bel_a:+}, B={bel_b:+}")
print(f"  Endstand: A={env.punkte_a}, B={env.punkte_b}")

---
## Teil 2: Der Q-Learning-Agent

### Grundidee: Lernen durch Erfahrung

Ein **Q-Learning-Agent** lernt, indem er Spiele spielt und aus den Ergebnissen schließt, welche Entscheidungen gut oder schlecht waren – ganz wie ein Mensch, der ein neues Brettspiel erlernt.

Er braucht dafür drei Dinge:

| Konzept | Was es bedeutet | Im Zahlenduell |
|---------|----------------|----------------|
| **Zustand** (State $s$) | Die aktuelle Spielsituation | Welche Karten habe ich noch? Welche hat der Gegner noch? |
| **Aktion** (Action $a$) | Die möglichen Entscheidungen | Welche Karte spiele ich? |
| **Belohnung** (Reward $r$) | Das Feedback der Umgebung | +1 / −1 pro Runde, +3 / −3 für den Gesamtsieg |

---

### Die Q-Tabelle: das Gedächtnis des Agenten

Der Agent führt eine **Q-Tabelle** – eine Art Nachschlagetabelle:

> **Q(s, a) = "Wie gut ist Aktion *a* im Zustand *s*?"**

Zu Beginn stehen überall Nullen. Mit jeder gespielten Runde werden die Einträge angepasst.
Am Ende des Trainings steht in der Tabelle die **gelernte Strategie** des Agenten.

**Beispiel** (nach dem Training, vereinfacht):

| Zustand | Karte 1 | Karte 2 | Karte 3 | Karte 4 | Karte 5 |
|---------|---------|---------|---------|---------|----------|
| Alle Karten vorhanden, Gegner hat noch alle | +0.12 | −0.05 | +0.18 | +0.09 | **+0.44** ★ |
| Nur noch 1 und 5 übrig, Gegner hat noch 1 und 5 | **+0.61** ★ | – | – | – | +0.22 |

★ = höchster Q-Wert → diese Aktion wählt der Agent

---

### Der Update: wie der Agent lernt

Nach jeder Runde wird der Q-Wert für das gespielte (Zustand, Aktion)-Paar aktualisiert:

$$Q(s, a) \;\leftarrow\; Q(s, a) + \alpha \cdot \underbrace{\Big[ r + \gamma \cdot \max_{a'} Q(s', a') - Q(s, a) \Big]}_{\text{TD-Fehler (Temporal Difference)}}$$

| Symbol | Name | Bedeutung |
|--------|------|-----------|
| $\alpha$ | Lernrate | Wie stark wirkt jedes neue Erlebnis? (typisch: 0.05–0.3) |
| $\gamma$ | Diskontfaktor | Wie wichtig sind *zukünftige* Belohnungen? (typisch: 0.9–0.99) |
| $r$ | Belohnung | Das direkte Feedback dieser Runde |
| $s'$ | Nachfolgezustand | Die Situation nach der Aktion |
| **TD-Fehler** | | Die Abweichung zwischen Erwartung und Realität – treibt das Lernen an |

**Intuition:** Der TD-Fehler ist wie die Überraschung nach einer Entscheidung. War das Ergebnis besser als erwartet → Q-Wert steigt. Schlechter als erwartet → Q-Wert sinkt.

---

### Erkunden oder Bewährtes nutzen? Das $\varepsilon$-greedy-Prinzip

Der Agent steht vor einem grundsätzlichen Dilemma:

- **Erkunden:** Zufällig handeln, um neue Situationen kennenzulernen  
- **Bewährtes nutzen:** Die bisher beste bekannte Aktion wählen

Lösung: **$\varepsilon$-greedy**. Mit Wahrscheinlichkeit $\varepsilon$ handelt der Agent zufällig, mit $1-\varepsilon$ wählt er den höchsten Q-Wert. Zu Beginn ist $\varepsilon = 1.0$ (alles Zufall), am Ende $\varepsilon \approx 0.05$ (fast immer Bewährtes nutzen).

---

### Self-Play: der Agent als sein eigener Trainingspartner

Beim **Self-Play** spielen **zwei Instanzen des Agenten gegeneinander** – beide lernen gleichzeitig. Das hat einen entscheidenden Vorteil: Je besser Agent A wird, desto anspruchsvoller wird der Gegner für Agent B, und umgekehrt. Das System steigert sich selbst.

Genau so haben AlphaGo und AlphaZero (DeepMind, 2016/2017) Go und Schach auf Weltklasseniveau erlernt – ausschließlich durch Self-Play, ohne je eine menschliche Partie zu analysieren.

In [ ]:
# ──────────────────────────────────────────────────────────────
# Interaktive Q-Tabellen-Demo
# Zeigt Schritt für Schritt, wie ein einzelner Q-Update funktioniert
# ──────────────────────────────────────────────────────────────

def zeige_q_update_beispiel():
    """
    Rechnet einen einzelnen Q-Learning-Update durch und
    visualisiert ihn als Tabelle + Erklärung.
    """
    # ── Beispielparameter ──
    alpha = 0.1      # Lernrate
    gamma = 0.95     # Diskontfaktor

    # Situation: Agent hat [1, 4, 5], Gegner hat [2, 3, 5]
    zustand     = ((1, 4, 5), (2, 3, 5))
    aktion      = 1          # Agent spielt Karte 1
    gegner_karte = 5         # Gegner spielt Karte 5  →  Sonderregel!
    belohnung   = 1          # +1 wegen Sonderregel (1 schlägt 5)

    # Folgezustand: Agent hat noch [4, 5], Gegner noch [2, 3]
    naechster_zustand = ((4, 5), (2, 3))
    naechste_karten   = [4, 5]

    # Fiktive Q-Tabelle (vor dem Update)
    q = {
        (zustand, 1): 0.10,
        (zustand, 4): 0.22,
        (zustand, 5): 0.35,
        (naechster_zustand, 4): 0.40,
        (naechster_zustand, 5): 0.18,
    }

    # ── Berechnung ──
    q_alt          = q[(zustand, aktion)]
    bester_folge_q = max(q[(naechster_zustand, a)] for a in naechste_karten)
    td_fehler      = belohnung + gamma * bester_folge_q - q_alt
    q_neu          = q_alt + alpha * td_fehler

    # ── Ausgabe ──
    sep = "─" * 55
    print(sep)
    print("  Q-Learning Update – Schritt für Schritt")
    print(sep)

    print(f"\n  Situation:")
    print(f"    Zustand s   = Ich habe {list(zustand[0])}, Gegner hat {list(zustand[1])}")
    print(f"    Aktion  a   = Karte {aktion} spielen")
    print(f"    Gegner      = spielt Karte {gegner_karte}")
    print(f"    Sonderregel = Abstand {abs(aktion - gegner_karte)} → kleinere Karte gewinnt!")
    print(f"    Belohnung r = {belohnung:+}")
    print(f"    Folge s'    = Ich habe {list(naechster_zustand[0])}, Gegner hat {list(naechster_zustand[1])}")

    print(f"\n  Q-Werte vor dem Update (Zustand s):")
    for k in naechste_karten + [aktion]:
        karte = list(zustand[0])[0] if k == aktion else k
    for karte in sorted(zustand[0]):
        marker = "  ← gespielt" if karte == aktion else ""
        print(f"    Q(s, Karte {karte}) = {q.get((zustand, karte), 0.0):.3f}{marker}")

    print(f"\n  Q-Werte im Folgezustand s':")
    for karte in sorted(naechster_zustand[0]):
        marker = "  ← max" if q[(naechster_zustand, karte)] == bester_folge_q else ""
        print(f"    Q(s', Karte {karte}) = {q[(naechster_zustand, karte)]:.3f}{marker}")

    print(f"\n  Berechnung:")
    print(f"    TD-Ziel    = r + γ · max Q(s', ·)")
    print(f"               = {belohnung} + {gamma} · {bester_folge_q:.3f}")
    print(f"               = {belohnung + gamma * bester_folge_q:.4f}")
    print(f"    TD-Fehler  = TD-Ziel − Q(s, a)")
    print(f"               = {belohnung + gamma * bester_folge_q:.4f} − {q_alt:.3f}")
    print(f"               = {td_fehler:.4f}")
    print(f"    Q_neu      = Q_alt + α · TD-Fehler")
    print(f"               = {q_alt:.3f} + {alpha} · {td_fehler:.4f}")
    print(f"               = {q_neu:.4f}")

    print(f"\n  Ergebnis:")
    print(f"    Q(s, Karte {aktion}): {q_alt:.3f}  →  {q_neu:.4f}  ({'+' if q_neu > q_alt else ''}{q_neu - q_alt:.4f})")
    print(f"    → Die Karte {aktion} wird in diesem Zustand etwas attraktiver bewertet.")
    print(sep)

    # ── Plot: Q-Werte vorher / nachher ──
    fig, ax = plt.subplots(figsize=(7, 3.5))
    karten   = sorted(zustand[0])
    q_vorher = [q.get((zustand, k), 0.0) for k in karten]
    q_nachher = list(q_vorher)          # Kopie
    idx_aktion = karten.index(aktion)
    q_nachher[idx_aktion] = q_neu

    x = range(len(karten))
    breite = 0.35
    b1 = ax.bar([i - breite/2 for i in x], q_vorher,  breite, label="vor Update",  color="#90CAF9", edgecolor="white")
    b2 = ax.bar([i + breite/2 for i in x], q_nachher, breite, label="nach Update", color="#1565C0", edgecolor="white")

    ax.bar([idx_aktion - breite/2], [q_vorher[idx_aktion]],   breite, color="#FFCC80", edgecolor="white")
    ax.bar([idx_aktion + breite/2], [q_nachher[idx_aktion]],  breite, color="#E65100", edgecolor="white",
           label=f"gespielte Karte ({aktion})")

    ax.set_xticks(list(x))
    ax.set_xticklabels([f"Karte {k}" for k in karten])
    ax.set_ylabel("Q-Wert")
    ax.set_title(f"Q-Werte für Zustand {zustand}\nvorher vs. nachher (α={alpha}, γ={gamma})")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    ax.set_ylim(0, max(q_nachher) * 1.3)
    plt.tight_layout()
    plt.show()


zeige_q_update_beispiel()

### Erkunden oder Bewährtes nutzen – interaktiv nachvollziehen

Die folgende Zelle simuliert, wie sich das $\varepsilon$-greedy-Verhalten bei verschiedenen $\varepsilon$-Werten auswirkt.
Ändert den Wert von `epsilon` und beobachtet, welche Karten gewählt werden.

In [ ]:
def epsilon_greedy_demo(epsilon=0.8, n_versuche=200):
    """
    Zeigt, wie ε-greedy zwischen Erkunden und Bewährtes-nutzen wechselt.
    
    Parameter:
        epsilon    : float  Wahrscheinlichkeit für zufällige Wahl (0.0 – 1.0)
        n_versuche : int    Wie oft soll eine Karte gewählt werden?
    """
    # Fiktive Q-Werte für fünf Karten
    q_werte = {1: 0.10, 2: 0.22, 3: 0.05, 4: 0.38, 5: 0.31}
    beste_karte = max(q_werte, key=q_werte.get)  # = Karte 4

    haeufigkeit = {k: 0 for k in q_werte}
    for _ in range(n_versuche):
        if random.random() < epsilon:
            wahl = random.choice(list(q_werte.keys()))   # Erkunden
        else:
            wahl = beste_karte                            # Bewährtes nutzen
        haeufigkeit[wahl] += 1

    # Ausgabe
    print(f"ε = {epsilon:.2f} | {n_versuche} Versuche")
    print(f"Beste bekannte Karte: {beste_karte} (Q={q_werte[beste_karte]})\n")
    for k, h in haeufigkeit.items():
        balken = "█" * int(h / n_versuche * 40)
        mark = " ← Bewährtes nutzen" if k == beste_karte else ""
        print(f"  Karte {k} ({h:3d}×):  {balken}{mark}")

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

    farben = ["#1565C0" if k == beste_karte else "#90CAF9" for k in q_werte]
    axes[0].bar(list(q_werte.keys()), list(q_werte.values()), color=farben, edgecolor="white")
    axes[0].set_title("Q-Werte (blau = beste Karte)")
    axes[0].set_xlabel("Karte")
    axes[0].set_ylabel("Q-Wert")
    axes[0].grid(axis="y", alpha=0.3)

    farben2 = ["#E65100" if k == beste_karte else "#FFCC80" for k in haeufigkeit]
    axes[1].bar(list(haeufigkeit.keys()), list(haeufigkeit.values()), color=farben2, edgecolor="white")
    axes[1].set_title(f"Gewählte Karten bei ε={epsilon:.2f}")
    axes[1].set_xlabel("Karte")
    axes[1].set_ylabel("Häufigkeit")
    axes[1].grid(axis="y", alpha=0.3)
    # Erwartungslinie
    erwartung_bewaehrt = n_versuche * (1 - epsilon + epsilon / len(q_werte))
    axes[1].axhline(erwartung_bewaehrt, color="#E65100", linestyle="--", alpha=0.6,
                    label=f"Erwartung beste Karte ({erwartung_bewaehrt:.0f}×)")
    axes[1].axhline(n_versuche * epsilon / len(q_werte), color="gray", linestyle=":", alpha=0.6,
                    label=f"Erwartung zufällige Karte ({n_versuche * epsilon / len(q_werte):.0f}×)")
    axes[1].legend(fontsize=9)

    plt.suptitle(f"ε-greedy: Erkunden={epsilon*100:.0f}%  |  Bewährtes nutzen={100-epsilon*100:.0f}%",
                 fontweight="bold")
    plt.tight_layout()
    plt.show()


# ── Hier den Wert ändern und Zelle erneut ausführen ──
epsilon_greedy_demo(epsilon=0.8)   # viel Erkunden
# epsilon_greedy_demo(epsilon=0.1) # fast nur Bewährtes nutzen
# epsilon_greedy_demo(epsilon=0.0) # nur Bewährtes nutzen

In [ ]:
class QLearningAgent:
    """
    Q-Learning Agent für Zahlenduell.
    Verwendet eine Q-Tabelle: {(zustand, aktion): q_wert}
    """

    def __init__(self, lernrate=0.1, diskont=0.95, epsilon=1.0, epsilon_min=0.05, epsilon_abfall=0.9995):
        self.q = defaultdict(float)   # Q-Tabelle
        self.alpha = lernrate         # Lernrate
        self.gamma = diskont          # Diskontfaktor
        self.epsilon = epsilon        # Zufallsquote (anfangs: viel zufällig erkunden)
        self.epsilon_min = epsilon_min
        self.epsilon_abfall = epsilon_abfall

    def aktion_waehlen(self, zustand, verfuegbare_karten):
        """Epsilon-greedy: mit Wahrscheinlichkeit epsilon zufällig erkunden, sonst Bewährtes nutzen."""
        if random.random() < self.epsilon:
            return random.choice(verfuegbare_karten)  # Erkunden
        # Bewährtes nutzen: wähle Karte mit höchstem Q-Wert
        return max(verfuegbare_karten, key=lambda a: self.q[(zustand, a)])

    def lernen(self, zustand, aktion, belohnung, naechster_zustand, naechste_karten, fertig):
        """Q-Learning Update."""
        if fertig or not naechste_karten:
            ziel = belohnung
        else:
            bester_naechster_q = max(self.q[(naechster_zustand, a)] for a in naechste_karten)
            ziel = belohnung + self.gamma * bester_naechster_q

        # TD-Update
        self.q[(zustand, aktion)] += self.alpha * (ziel - self.q[(zustand, aktion)])

        # Epsilon langsam reduzieren
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_abfall

    @property
    def q_tabelle_groesse(self):
        return len(self.q)


print("✅ Q-Learning Agent definiert")

---
## Teil 3: Self-Play Training

Zwei Agenten spielen **gegeneinander** und lernen gleichzeitig. Kein Mensch gibt vor, was gut oder schlecht ist – die Agenten entdecken Strategien selbst.

In [ ]:
def trainieren(episoden=50000, log_intervall=1000):
    """
    Trainiert zwei Q-Learning Agenten im Self-Play.
    Returns: agent_a, agent_b, trainings_log
    """
    agent_a = QLearningAgent()
    agent_b = QLearningAgent()
    env = ZahlenduellUmgebung()

    log = {"siege_a": [], "siege_b": [], "unentschieden": [], "epsilon": [], "q_groesse": []}
    siege_a = siege_b = unentschieden = 0

    for episode in range(1, episoden + 1):
        za, zb = env.reset()

        while True:
            # Aktionen wählen
            ka = agent_a.aktion_waehlen(za, env.karten_a)
            kb = agent_b.aktion_waehlen(zb, env.karten_b)

            # Schritt
            za_neu, zb_neu, bel_a, bel_b, fertig = env.schritt(ka, kb)

            # Lernen
            agent_a.lernen(za, ka, bel_a, za_neu, env.karten_a, fertig)
            agent_b.lernen(zb, kb, bel_b, zb_neu, env.karten_b, fertig)

            za, zb = za_neu, zb_neu

            if fertig:
                if env.punkte_a > env.punkte_b:
                    siege_a += 1
                elif env.punkte_b > env.punkte_a:
                    siege_b += 1
                else:
                    unentschieden += 1
                break

        # Log alle N Episoden
        if episode % log_intervall == 0:
            total = siege_a + siege_b + unentschieden
            log["siege_a"].append(siege_a / total * 100)
            log["siege_b"].append(siege_b / total * 100)
            log["unentschieden"].append(unentschieden / total * 100)
            log["epsilon"].append(agent_a.epsilon)
            log["q_groesse"].append(agent_a.q_tabelle_groesse)
            siege_a = siege_b = unentschieden = 0

    print(f"✅ Training abgeschlossen!")
    print(f"   Q-Tabelle Agent A: {agent_a.q_tabelle_groesse} Einträge")
    print(f"   Epsilon (final):   {agent_a.epsilon:.4f}")
    return agent_a, agent_b, log


print("Starte Training mit 50.000 Episoden...")
agent_a, agent_b, log = trainieren(episoden=50000)


---
## Teil 4: Lernkurve visualisieren

In [ ]:
def plot_lernkurve(log, log_intervall=1000):
    episoden = [i * log_intervall for i in range(1, len(log["siege_a"]) + 1)]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle("Zahlenduell – Self-Play Training", fontsize=14, fontweight="bold")

    # --- Plot 1: Siegquoten ---
    ax = axes[0]
    ax.plot(episoden, log["siege_a"], label="Agent A", color="#2196F3", linewidth=2)
    ax.plot(episoden, log["siege_b"], label="Agent B", color="#FF5722", linewidth=2)
    ax.plot(episoden, log["unentschieden"], label="Unentschieden", color="gray",
            linewidth=1.5, linestyle="--")
    ax.axhline(y=33.3, color="gray", linestyle=":", alpha=0.5, label="Zufall (33%")
    ax.set_xlabel("Episode")
    ax.set_ylabel("Siegquote (%)")
    ax.set_title("Siegquoten (pro 1000 Spiele)")
    ax.legend()
    ax.set_ylim(0, 80)
    ax.grid(alpha=0.3)

    # --- Plot 2: Epsilon (Zufallsquote) ---
    ax = axes[1]
    ax.plot(episoden, [e * 100 for e in log["epsilon"]], color="#4CAF50", linewidth=2)
    ax.fill_between(episoden, [e * 100 for e in log["epsilon"]], alpha=0.2, color="#4CAF50")
    ax.set_xlabel("Episode")
    ax.set_ylabel("Epsilon (%)")
    ax.set_title("Erkunden → Bewährtes nutzen")
    ax.grid(alpha=0.3)

    # --- Plot 3: Q-Tabellen-Wachstum ---
    ax = axes[2]
    ax.plot(episoden, log["q_groesse"], color="#9C27B0", linewidth=2)
    ax.set_xlabel("Episode")
    ax.set_ylabel("Einträge in Q-Tabelle")
    ax.set_title("Wachstum der Q-Tabelle")
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


plot_lernkurve(log)

---
## Teil 5: Was hat der Agent gelernt?

Jetzt schauen wir, welche Karten der trainierte Agent in bestimmten Situationen bevorzugt.

In [ ]:
def zeige_strategie(agent, situationen):
    """
    Zeigt für ausgewählte Situationen die Q-Werte aller möglichen Aktionen.
    """
    print("Gelernte Strategie des Agenten\n" + "=" * 50)

    for eigene_karten, gegner_karten in situationen:
        zustand = (tuple(sorted(eigene_karten)), tuple(sorted(gegner_karten)))
        print(f"\nEigene Karten: {sorted(eigene_karten)}")
        print(f"Gegnerkarten (noch vorhanden): {sorted(gegner_karten)}")

        q_werte = {k: agent.q[(zustand, k)] for k in eigene_karten}
        beste = max(q_werte, key=q_werte.get)

        for karte in sorted(eigene_karten):
            markierung = " ← BEST" if karte == beste else ""
            balken = "█" * max(0, int((q_werte[karte] + 5) * 4))
            print(f"  Karte {karte}: Q={q_werte[karte]:+.3f}  {balken}{markierung}")


# Interessante Situationen analysieren
situationen = [
    # (eigene Karten, Gegnerkarten)
    ([1, 2, 3, 4, 5], [1, 2, 3, 4, 5]),     # Spielanfang
    ([1, 5],          [1, 5]),                # Endspiel mit Sonderregel-Potenzial
    ([1, 2, 3],       [3, 4, 5]),             # Gegner hat nur hohe Karten
    ([1],             [5]),                   # Letzte Karte – Sonderregel!
]

zeige_strategie(agent_a, situationen)

---
## Teil 6: Agent vs. naive Strategie

Wie gut ist unser trainierter Agent gegen einen Gegner, der immer die **höchste Karte** spielt?

> 💡 **Zwei Stolpersteine bei dieser Auswertung:**
>
> **1. Determinismus:** Sowohl der greedy-Agent als auch der naive Gegner spielen ohne jeden Zufall. Bei gleichem Startzustand kommt deshalb *immer* derselbe Spielverlauf heraus -- egal wie oft man testet. Mit `test_epsilon=0` läge das Ergebnis bei *jeder* Anzahl an Partien exakt bei z.B. 100/0/0, weil effektiv immer dieselbe einzelne Partie wiederholt wird. Unten lassen wir dem Agenten daher eine kleine Rest-Wahrscheinlichkeit (`test_epsilon`), gelegentlich eine andere Karte zu spielen -- das erzeugt echte Variation zwischen den Partien.
>
> **2. Self-Play lernt keinen Konter gegen einen bestimmten Gegner, sondern eine robuste Allzweck-Strategie.** Das kann überraschen: Es *gibt* eine perfekte Antwort auf "immer höchste Karte" (Karte 1 zuerst spielen, um die gegnerische 5 über die Sonderregel zu schlagen, danach aufsteigend weiterspielen -- damit gewinnt man alle 5 Runden). Unser Agent hat diesen speziellen Trick aber nicht zwangsläufig gelernt, weil er nie gegen *diesen* Gegnertyp trainiert hat, sondern nur gegen einen anderen, ebenfalls lernenden Agenten. Self-Play produziert eine Strategie, die gegen wechselnde, anpassungsfähige Gegner gut funktioniert -- nicht notwendigerweise die mathematisch optimale Antwort auf einen einzigen, starren Spielstil.

In [ ]:
def test_gegen_naive(agent, anzahl=2000, test_epsilon=0.1):
    """
    Testet den trainierten Agenten (als A) gegen einen naiven Gegner (immer höchste Karte).
    
    WICHTIG: Sowohl der greedy Agent als auch der naive Gegner spielen vollkommen
    deterministisch (keine Zufallskomponente). Würden wir test_epsilon=0 setzen,
    liefe JEDE der `anzahl` Partien exakt identisch ab -- das Ergebnis wäre dann
    immer 100/0/0 (oder 0/100/0 etc.), unabhängig von `anzahl`. Das ist kein Fehler
    der Simulation, sondern eine Eigenschaft deterministischer Strategien: es gibt
    nur EINEN möglichen Spielverlauf, der sich beliebig oft wiederholt.
    
    Mit `test_epsilon` geben wir dem Agenten eine kleine Rest-Wahrscheinlichkeit,
    auch eine suboptimale Karte zu spielen. Das erzeugt echte Variation zwischen
    den Partien und macht die Siegquote zu einer aussagekräftigen Kennzahl.
    """
    env = ZahlenduellUmgebung()
    siege_agent = siege_naive = unentschieden = 0

    for _ in range(anzahl):
        za, zb = env.reset()

        while True:
            # Agent: meistens greedy, mit kleiner Wahrscheinlichkeit zufällig
            if random.random() < test_epsilon:
                ka = random.choice(env.karten_a)
            else:
                ka = max(env.karten_a, key=lambda a: agent.q[(za, a)])
            # Naiver Gegner
            kb = max(env.karten_b)

            za, zb, _, _, fertig = env.schritt(ka, kb)

            if fertig:
                if env.punkte_a > env.punkte_b:
                    siege_agent += 1
                elif env.punkte_b > env.punkte_a:
                    siege_naive += 1
                else:
                    unentschieden += 1
                break

    print("Agent (RL) vs. Naiver Gegner (immer höchste Karte)")
    print(f"  Spiele gesamt:     {anzahl}  (Test-ε={test_epsilon})")
    print(f"  Siege Agent (RL):  {siege_agent:4d} ({siege_agent/anzahl*100:.1f}%)")
    print(f"  Siege Naive:       {siege_naive:4d} ({siege_naive/anzahl*100:.1f}%)")
    print(f"  Unentschieden:     {unentschieden:4d} ({unentschieden/anzahl*100:.1f}%)")

    # Balkendiagramm
    labels = ["RL-Agent", "Naive\n(höchste Karte)", "Unentschieden"]
    werte = [siege_agent / anzahl * 100, siege_naive / anzahl * 100, unentschieden / anzahl * 100]
    farben = ["#2196F3", "#FF5722", "gray"]

    plt.figure(figsize=(7, 4))
    balken = plt.bar(labels, werte, color=farben, width=0.5, edgecolor="white")
    for b, w in zip(balken, werte):
        plt.text(b.get_x() + b.get_width() / 2, w + 0.5, f"{w:.1f}%",
                 ha="center", va="bottom", fontweight="bold")
    plt.ylabel("Siegquote (%)")
    plt.title("RL-Agent vs. Naive Strategie")
    plt.ylim(0, 90)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


test_gegen_naive(agent_a)

### Bonus: Gibt es eine perfekte Antwort auf den naiven Gegner?

Da der naive Gegner *deterministisch* immer die höchste Karte spielt, können wir die optimale Gegenstrategie sogar exakt berechnen (Brute-Force über alle 5! = 120 möglichen Kartenreihenfolgen). Das zeigt, wie nah unser Self-Play-Agent an das theoretische Optimum herankommt -- bzw. wo Self-Play an seine Grenzen stößt.

In [ ]:
from itertools import permutations

def finde_optimale_gegenstrategie():
    """
    Berechnet per Brute-Force die beste Kartenreihenfolge gegen einen
    Gegner, der garantiert 5,4,3,2,1 spielt (immer die höchste verbliebene Karte).
    """
    naive_folge = [5, 4, 3, 2, 1]

    bester_score = -999
    beste_reihenfolge = None
    for perm in permutations([1, 2, 3, 4, 5]):
        score = sum(bewerte_runde(perm[i], naive_folge[i]) for i in range(5))
        if score > bester_score:
            bester_score = score
            beste_reihenfolge = perm

    return beste_reihenfolge, bester_score


beste_reihenfolge, bester_score = finde_optimale_gegenstrategie()
print(f"Optimale Reihenfolge gegen 'immer höchste Karte': {beste_reihenfolge}")
print(f"Erreichbarer Score: {bester_score} von 5 möglichen Rundensiegen\n")

print("Spielverlauf:")
for i, (eigene, gegner) in enumerate(zip(beste_reihenfolge, [5, 4, 3, 2, 1]), start=1):
    ergebnis = bewerte_runde(eigene, gegner)
    text = "gewinne" if ergebnis == 1 else ("verliert" if ergebnis == -1 else "Unentschieden")
    sonder = " (Sonderregel!)" if abs(eigene - gegner) == 4 else ""
    print(f"  Runde {i}: Ich spiele {eigene}, Gegner spielt {gegner} → ich {text}{sonder}")

print(f"\nUnser Self-Play-Agent spielt im Startzustand stattdessen Karte "
      f"{max(range(1,6), key=lambda a: agent_a.q[((1,2,3,4,5),(1,2,3,4,5)), a])}.")
print("→ Self-Play hat eine andere, gegen wechselnde Gegner robuste Strategie gefunden --")
print("  nicht zwingend die spezifisch optimale Antwort auf diesen einen starren Gegnertyp.")

---
## Teil 7: Interaktiv gegen den Agenten spielen

In [ ]:
def spiele_gegen_agent(agent):
    """
    Mensch (A) gegen trainierten Agenten (B).
    """
    env = ZahlenduellUmgebung()
    za, zb = env.reset()

    print("=" * 45)
    print("     ZAHLENDUELL – Du vs. KI-Agent")
    print("=" * 45)
    print("Sonderregel: Abstand von 4 → kleinere Zahl gewinnt!\n")

    punkte_du = 0
    punkte_agent = 0

    for runde in range(1, 6):
        print(f"--- Runde {runde} ---")
        print(f"Deine Karten:  {sorted(env.karten_a)}")
        print(f"Agent hat noch: {len(env.karten_b)} Karte(n)")

        # Spieler wählt
        while True:
            try:
                wahl = int(input(f"Welche Karte spielst du? {sorted(env.karten_a)}: "))
                if wahl in env.karten_a:
                    break
                print("Diese Karte hast du nicht mehr!")
            except ValueError:
                print("Bitte eine Zahl eingeben.")

        # Agent wählt (greedy)
        agent_wahl = max(env.karten_b, key=lambda a: agent.q[(zb, a)])

        za, zb, bel_a, bel_b, fertig = env.schritt(wahl, agent_wahl)

        print(f"Du: {wahl}  |  Agent: {agent_wahl}")
        if bel_a > 0:
            punkte_du += 1
            print("➡️  Du gewinnst die Runde!")
        elif bel_b > 0:
            punkte_agent += 1
            print("➡️  Agent gewinnt die Runde!")
        else:
            print("➡️  Unentschieden!")
        print(f"Stand: Du {punkte_du} – Agent {punkte_agent}\n")

    print("=" * 45)
    if punkte_du > punkte_agent:
        print("🏆 Du hast gewonnen! (Beeindruckend!)")
    elif punkte_agent > punkte_du:
        print("🤖 Der Agent hat gewonnen!")
    else:
        print("🤝 Unentschieden!")


# Auskommentieren zum Spielen:
# spiele_gegen_agent(agent_b)  # Du spielst gegen agent_b
print("💡 Entferne das '#' in der letzten Zeile und führe die Zelle aus, um gegen den Agenten zu spielen!")

---
## Teil 8: Experimente und Diskussion

### 🔬 Aufgaben zum Ausprobieren

1. **Lernrate verändern:** Ändere `lernrate` auf 0.5 oder 0.01 – wie verändert sich die Lernkurve?

2. **Weniger Training:** Trainiere nur 5.000 Episoden. Ist der Agent noch gut?

3. **Belohnungsstruktur:** Was passiert, wenn du die Endbelohnung (±3) erhöhst oder entfernst?

4. **Sonderregel entfernen:** Entferne die Sonderregel in `bewerte_runde`. Was ändert sich an der Strategie?

5. **Zustandsraum:** Wie viele verschiedene Zustände gibt es theoretisch maximal? *(Tipp: Jeder Spieler kann Teilmengen von {1..5} übrig haben)*

### 💬 Diskussionsfragen

- Warum braucht der Agent so viele Episoden? Was wäre bei einem einfacheren Spiel anders?
- Wie unterscheidet sich Self-Play von "Training gegen einen fixen Gegner"?
- Welche realen Systeme nutzen ähnliche Prinzipien? *(Hinweis: Schach, Go, autonomes Fahren...)*
- Was ist der Nachteil einer Q-Tabelle bei komplexeren Spielen?

In [ ]:
# Bonus: Zustandsraum-Analyse
from itertools import combinations

alle_moeglichen_haende = []
for groesse in range(0, 6):
    for combo in combinations(range(1, 6), groesse):
        alle_moeglichen_haende.append(combo)

alle_zustaende = [(h_eigen, h_gegner)
                  for h_eigen in alle_moeglichen_haende
                  for h_gegner in alle_moeglichen_haende
                  if len(h_eigen) == len(h_gegner)]  # Spieler haben immer gleich viele Karten

print(f"Mögliche Zustände (inkl. leere Hand): {len(alle_zustaende)}")
print(f"Max. mögliche Q-Einträge (Zustände × Aktionen): {sum(len(z[0]) for z in alle_zustaende)}")
print(f"Tatsächliche Q-Einträge nach Training: {agent_a.q_tabelle_groesse}")
print()
print("→ Nicht alle Zustände wurden erkundet – das ist normal!")